In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor
from data_functions import *

sales_dirty = pd.read_csv(PATHS["sales_train"])
stock_dirty = pd.read_csv(PATHS["stock_train"])

# EDA

## Вид данных

In [2]:
sales_dirty.head(3)

,date,composite_division,cities_name,warehouse_name,category_lvl_1,category_lvl_2,category_lvl_3,article,characteristic,quantity,sales_sum,discount_sum
0,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_0,category_lvl_2_0,NaN,char_article_0,characteristic_0,2.0,7.429034,5.459302
1,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_1,category_lvl_2_1,NaN,char_article_1,characteristic_0,1.0,9.613652,7.798864
2,2024-01-01,composite_division_0,cities_name_0,warehouse_name_0,category_lvl_1_2,category_lvl_2_2,NaN,char_article_2,characteristic_0,1.0,9.237081,7.258904


In [3]:
stock_dirty.head(3)

,date,warehouse_name,article,stock
0,2023-12-13,warehouse_name_0,char_article_119,1
1,2023-12-13,warehouse_name_3,char_article_62,135
2,2023-12-13,warehouse_name_3,char_article_268,3


In [4]:
print(f"Размер sales_dirty: {sales_dirty.shape}")
print(f"Размер stock_dirty: {stock_dirty.shape}")

Размер sales_dirty: (1492370, 12)
Размер stock_dirty: (6153988, 4)


In [5]:
sales_dirty.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1492370 entries, 0 to 1492369
Data columns (total 12 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   date                1492370 non-null  object 
 1   composite_division  1492370 non-null  object 
 2   cities_name         1492370 non-null  object 
 3   warehouse_name      1492370 non-null  object 
 4   category_lvl_1      1492370 non-null  object 
 5   category_lvl_2      1492370 non-null  object 
 6   category_lvl_3      938 non-null      object 
 7   article             1492370 non-null  object 
 8   characteristic      1492370 non-null  object 
 9   quantity            1492370 non-null  float64
 10  sales_sum           1492367 non-null  float64
 11  discount_sum        1175415 non-null  float64
dtypes: float64(3), object(9)
memory usage: 136.6+ MB


In [6]:
stock_dirty.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6153988 entries, 0 to 6153987
Data columns (total 4 columns):
 #   Column          Dtype 
---  ------          ----- 
 0   date            object
 1   warehouse_name  object
 2   article         object
 3   stock           int64 
dtypes: int64(1), object(3)
memory usage: 187.8+ MB


In [16]:
sales_dirty.describe()

,quantity,sales_sum,discount_sum
count,1.492370e+06,1.492367e+06,1.175415e+06
mean,1.224087e+00,8.763223e+00,2.962457e+00
std,7.942246e-01,8.737589e-01,3.524857e+00
min,1.000000e+00,-1.340941e-01,-5.863008e+00
25%,1.000000e+00,8.300911e+00,-6.201960e-03
50%,1.000000e+00,8.863898e+00,4.893390e-02
75%,1.000000e+00,9.330486e+00,6.767874e+00
max,2.000000e+02,1.321461e+01,1.312170e+01


## Уникальные значения 

In [7]:
print("sales_dirty:", sales_dirty.nunique(), sep="\n", end="\n\n")
print("sales_dirty:", stock_dirty.nunique(), sep="\n", end="\n\n")

sales_dirty:
date                      943
composite_division         16
cities_name               209
warehouse_name             11
category_lvl_1             14
category_lvl_2             51
category_lvl_3              1
article                   799
characteristic             61
quantity                   59
sales_sum             1492367
discount_sum          1175415
dtype: int64

sales_dirty:
date              962
warehouse_name     11
article           799
stock             884
dtype: int64



In [8]:
print(sales_dirty[sales_dirty["category_lvl_3"].notna()].nunique())
sales_dirty[sales_dirty["category_lvl_3"].notna()].head()


date                  555
composite_division     11
cities_name           151
warehouse_name         11
category_lvl_1          1
category_lvl_2          1
category_lvl_3          1
article                 1
characteristic          1
quantity                2
sales_sum             938
discount_sum          781
dtype: int64


,date,composite_division,cities_name,warehouse_name,category_lvl_1,category_lvl_2,category_lvl_3,article,characteristic,quantity,sales_sum,discount_sum
5938,2024-01-14,composite_division_1,cities_name_32,warehouse_name_3,category_lvl_1_1,category_lvl_2_45,category_lvl_3_0,char_article_396,characteristic_0,1.0,8.728841,-0.118590
8867,2024-01-20,composite_division_5,cities_name_20,warehouse_name_2,category_lvl_1_1,category_lvl_2_45,category_lvl_3_0,char_article_396,characteristic_0,1.0,8.726030,0.058334
9819,2024-01-22,composite_division_1,cities_name_57,warehouse_name_5,category_lvl_1_1,category_lvl_2_45,category_lvl_3_0,char_article_396,characteristic_0,1.0,8.728847,-0.008115
10066,2024-01-23,composite_division_0,cities_name_4,warehouse_name_2,category_lvl_1_1,category_lvl_2_45,category_lvl_3_0,char_article_396,characteristic_0,1.0,8.864182,6.370806
11199,2024-01-25,composite_division_1,cities_name_69,warehouse_name_2,category_lvl_1_1,category_lvl_2_45,category_lvl_3_0,char_article_396,characteristic_0,1.0,8.727664,-0.037007


## Ненулевые значения

In [9]:
print("sales_dirty:", sales_dirty.isna().sum(), sep="\n", end="\n\n")
print("stock_dirty:", stock_dirty.isna().sum(), sep="\n", end="\n\n")

sales_dirty:
date                        0
composite_division          0
cities_name                 0
warehouse_name              0
category_lvl_1              0
category_lvl_2              0
category_lvl_3        1491432
article                     0
characteristic              0
quantity                    0
sales_sum                   3
discount_sum           316955
dtype: int64

stock_dirty:
date              0
warehouse_name    0
article           0
stock             0
dtype: int64



## TODO-ШКИ
* TODO_1: date -> datetime 
* TODO_2: проверить, совпадают ли множества айдишников warehouse_name & article в двух датасетах
* TODO_3: composite_division, cities_name, warehouse_name - это id, их нужно кодировать, группировать по ним или, возможно, аггрегировать. ПО [П.3 ТЗ](../markdown%20hakaton.md#stock_dataset_hakatoncsv--остатки-на-складах-по-дням) - все департаменты суммируются в рамках пары склад - товар, то есть в прогнозе не нужно разделять по `composite_division`
* TODO_4: category_lvl_N - категории товаров типа Электроника-бытовая техника-кофемашины (хотя в category_lvl_3 всего одно уникальное значение).
* TODO_5: article - для sales то же самое, что и char_article для stock. тоже id, агрегация для товаров, target encoding
* TODO_6: characteristic я даже не знаю, что с этим делать. если это буквально описание товара, то оно будто бы вообще не нужно нам
* TODO_7: quantity - ТАРГЕТ, который нужно предсказать
* TODO_8: sales_sum и discount_sum - цена и скидка, наверняка ОБА очень сильно коррелируют с таргетом, потому что размер скидки важен и сама цена тоже, ПОСМОТРЕТЬ ЗАВИСИМОСТИ. на тестовой выборке их не будет. возможно, стоит ввести понятие тренда за последний месяц, как за историческую фичу, сколько продано определённого товара в среднем

* TODO_9: category_lvl_3 всего одно значение, остальные NaN. при этом это одно-единственное значение только с category_lvl_1_1, category_lvl_2_45. стоит заполнить другой категорией типа other или другие.
* TODO_10: проверить непрерывность временного диапазона, в sales_dirty больше данных, чем в stock_dirty.
* TODO_11: 799 товаров * 11 складов = 8789 комбинаций. 943 временных дат в stock * 8789 = 8.28млн строк, у нас всеего 1.49 строк. значит, большинство товаров продаются не каждый день. может, имеет смысл ввести двухэтапное предсказание: первая модель предсказывает, будет ли продаваться товар сегодня, а вторая уже на основе этого предсказания предсказывает конечное количество товаров, ну и соответственно ввести исторический признак, сколько уже времени товар не продаётся
* TODO_12: как часто случается, что заканчивается товар
* TODO_13: как часто случается, что товара нет, от чего нет продаж, по отношению к отсутствию спроса -> ввести что-то по типу спроса на товар (вероятность, что купят из предположения о двух моделях?)
* TODO_14: проверить, закреплена ли за каждым товаром определённая категория и характеристика, закреплен ли за складом город
* TODO_15: ГДЕ КОЛОНКА volume_weight????? В md ОНА ЕСТЬ, А ЗДЕСЬ НЕМА?!
* TODO_16: 1.5 млн значений quantity и всего 59 значений, во прикол. 
* TODO_17: придумать агрегат для cities_name - склад отгружает во много городов, при схлопывании пар склад-товар город теряется.
* TODO_18: анализ жизненного цикла товаров - проверить последние даты продаж и остатков по парам (склад, товар). Отсечь товары, которые уже выведены из ассортимента или ввести признак

In [ ]:
# TODO_2
print(set(stock_dirty['article']) == set(sales_dirty['article']))
set(stock_dirty['warehouse_name']) == set(sales_dirty['warehouse_name'])
# множества айдишников warehouse_name & article совпадают в двух датасетах

True


True

In [ ]:
# TODO_14
print(sales_dirty.groupby('article')[['category_lvl_1', 'category_lvl_2']].nunique().max(), end="\n\n")
print('characteristic\t ', sales_dirty.groupby('article')['characteristic'].nunique().max(), end="\n\n")
print(sales_dirty.groupby('warehouse_name')['cities_name'].nunique())
# каждому товару соответствует категория и характеристика
# склады распределяются на несколько городов

category_lvl_1    1
category_lvl_2    1
dtype: int64

characteristic	  1

warehouse_name
warehouse_name_0      3
warehouse_name_1     17
warehouse_name_10    16
warehouse_name_2     80
warehouse_name_3     19
warehouse_name_4      9
warehouse_name_5     11
warehouse_name_6     13
warehouse_name_7     14
warehouse_name_8      8
warehouse_name_9     19
Name: cities_name, dtype: int64
